# 00 — Data Ingestion v1: External Data Sources

Downloads non-OHLCV data that cannot be derived from Binance candles:

| Phase | Source | Resolution | Output |
|-------|--------|-----------|--------|
| A | Fear & Greed Index (alternative.me) | Daily | `data/external/fear_greed_index.parquet` |
| B | Per-coin market caps (CoinGecko) | Daily | `data/external/coingecko_market_caps.parquet` |
| C | Approx. historical market caps (OHLCV × supply) | Daily | `data/external/approx_market_caps.parquet` |

**Prerequisites:** Run `00_data_ingestion_v0.ipynb` first (Binance OHLCV parquets must exist in `data/raw/`).

**Note:** CoinGecko free tier allows ~30 calls/min. The script sleeps 12s between coins to stay safe.

In [1]:
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

pandas 3.0.2  |  numpy 2.4.4


In [2]:
def _find_repo_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("pyproject.toml not found; run from within the repo")

REPO = _find_repo_root()
RAW_DIR = REPO / "data" / "raw"
EXT_DIR = REPO / "data" / "external"
STATIC_DIR = REPO / "data" / "static"
EXT_DIR.mkdir(parents=True, exist_ok=True)

SYMBOLS_1H = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "SOLUSDT",
    "ADAUSDT", "DOGEUSDT", "AVAXUSDT", "DOTUSDT", "LINKUSDT",
]

COINGECKO_IDS = {
    "BTCUSDT": "bitcoin",
    "ETHUSDT": "ethereum",
    "BNBUSDT": "binancecoin",
    "XRPUSDT": "ripple",
    "SOLUSDT": "solana",
    "ADAUSDT": "cardano",
    "DOGEUSDT": "dogecoin",
    "AVAXUSDT": "avalanche-2",
    "DOTUSDT": "polkadot",
    "LINKUSDT": "chainlink",
    "USDTUSDT": "tether",
    "USDCUSDT": "usd-coin",
}

print(f"REPO      : {REPO}")
print(f"RAW_DIR   : {RAW_DIR}")
print(f"EXT_DIR   : {EXT_DIR}")

REPO      : /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system
RAW_DIR   : /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/raw
EXT_DIR   : /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/external


## Phase A — Fear & Greed Index

- **API:** `GET https://api.alternative.me/fng/?limit=0` (free, no key)
- **History:** Feb 2018 → present (~3,000 daily points)
- **Value:** 0 (Extreme Fear) → 100 (Extreme Greed)

In [3]:
FNG_URL = "https://api.alternative.me/fng/"
FNG_OUTPUT = EXT_DIR / "fear_greed_index.parquet"

resp = requests.get(FNG_URL, params={"limit": "0"}, timeout=30)
resp.raise_for_status()
fng_data = resp.json()["data"]

fng_df = pd.DataFrame(fng_data)
fng_df["value"] = fng_df["value"].astype(int)
fng_df["timestamp"] = pd.to_datetime(fng_df["timestamp"].astype(int), unit="s", utc=True)
fng_df = fng_df.rename(columns={"timestamp": "date", "value_classification": "classification"})
fng_df = fng_df[["date", "value", "classification"]].sort_values("date").reset_index(drop=True)
fng_df = fng_df.set_index("date")
if "time_until_update" in fng_df.columns:
    fng_df = fng_df.drop(columns=["time_until_update"])

fng_df.to_parquet(FNG_OUTPUT)

print(f"Downloaded {len(fng_df)} daily records")
print(f"Range: {fng_df.index.min().date()} -> {fng_df.index.max().date()}")
print(f"Value range: {fng_df['value'].min()} - {fng_df['value'].max()}")
print(f"Saved: {FNG_OUTPUT}")
fng_df.tail(10)

Downloaded 3035 daily records
Range: 2018-02-01 -> 2026-05-28
Value range: 5 - 95
Saved: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/external/fear_greed_index.parquet


,value,classification
date,,
2026-05-19 00:00:00+00:00,25,Extreme Fear
2026-05-20 00:00:00+00:00,27,Fear
2026-05-21 00:00:00+00:00,29,Fear
2026-05-22 00:00:00+00:00,28,Fear
2026-05-23 00:00:00+00:00,28,Fear
2026-05-24 00:00:00+00:00,25,Extreme Fear
2026-05-25 00:00:00+00:00,30,Fear
2026-05-26 00:00:00+00:00,34,Fear
2026-05-27 00:00:00+00:00,25,Extreme Fear


## Phase B — CoinGecko Market Caps (~1 year daily)

- **API:** `GET /coins/{id}/market_chart?vs_currency=usd&days=365`
- **Coins:** 10 trading pairs + USDT + USDC (12 total)
- **Rate limit:** 12s sleep between calls (free tier)

In [4]:
CG_BASE = "https://api.coingecko.com/api/v3"
CG_OUTPUT = EXT_DIR / "coingecko_market_caps.parquet"
CG_DAYS = 365

all_mcap_records = []

for symbol, cg_id in COINGECKO_IDS.items():
    print(f"  Fetching {cg_id} ({symbol})...", end=" ", flush=True)
    try:
        resp = requests.get(
            f"{CG_BASE}/coins/{cg_id}/market_chart",
            params={"vs_currency": "usd", "days": str(CG_DAYS)},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()

        prices = data.get("prices", [])
        mcaps = data.get("market_caps", [])
        volumes = data.get("total_volumes", [])

        n = min(len(prices), len(mcaps), len(volumes))
        for i in range(n):
            ts = pd.Timestamp(prices[i][0], unit="ms", tz="UTC")
            all_mcap_records.append({
                "date": ts,
                "symbol": symbol,
                "cg_id": cg_id,
                "price": prices[i][1],
                "market_cap": mcaps[i][1] if mcaps[i][1] else np.nan,
                "total_volume": volumes[i][1] if volumes[i][1] else np.nan,
            })
        print(f"{n} records")
        time.sleep(12)
    except Exception as e:
        print(f"ERROR: {e}")
        time.sleep(15)

cg_df = pd.DataFrame(all_mcap_records)
cg_df.to_parquet(CG_OUTPUT, index=False)

print(f"\nTotal records: {len(cg_df)}")
print(f"Coins: {cg_df['symbol'].nunique()}")
print(f"Date range: {cg_df['date'].min().date()} -> {cg_df['date'].max().date()}")
print(f"Saved: {CG_OUTPUT}")

  Fetching bitcoin (BTCUSDT)... 366 records
  Fetching ethereum (ETHUSDT)... 366 records
  Fetching binancecoin (BNBUSDT)... 366 records
  Fetching ripple (XRPUSDT)... 366 records
  Fetching solana (SOLUSDT)... 366 records
  Fetching cardano (ADAUSDT)... 366 records
  Fetching dogecoin (DOGEUSDT)... 366 records
  Fetching avalanche-2 (AVAXUSDT)... 366 records
  Fetching polkadot (DOTUSDT)... 366 records
  Fetching chainlink (LINKUSDT)... 366 records
  Fetching tether (USDTUSDT)... 366 records
  Fetching usd-coin (USDCUSDT)... 366 records

Total records: 4392
Coins: 12
Date range: 2025-05-29 -> 2026-05-28
Saved: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/external/coingecko_market_caps.parquet


## Phase C — Approximate Historical Market Caps

For dates before the CoinGecko 1-year window:

```
market_cap(coin, t) = close_price(t) * circulating_supply
```

Uses static supply from `data/static/crypto_market_caps.csv`. Approximate (supply changes over time) but captures 90%+ of dominance variance.

In [5]:
APPROX_OUTPUT = EXT_DIR / "approx_market_caps.parquet"
SUPPLY_PATH = STATIC_DIR / "crypto_market_caps.csv"

supply_df = pd.read_csv(SUPPLY_PATH)
supply_map = dict(zip(supply_df["symbol"], supply_df["circulating_supply"]))
print(f"Loaded circulating supply for {len(supply_map)} coins")

approx_records = []

for symbol in SYMBOLS_1H:
    parquet_path = RAW_DIR / f"{symbol}_1h.parquet"
    if not parquet_path.exists():
        print(f"  {symbol}: OHLCV not found, skipping")
        continue

    df = pd.read_parquet(parquet_path)
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    supply = supply_map.get(symbol, np.nan)
    if np.isnan(supply):
        print(f"  {symbol}: no supply data, skipping")
        continue

    daily = df["close"].resample("1D").last().dropna()
    daily_mcap = daily * supply

    for ts, mcap in daily_mcap.items():
        approx_records.append({
            "date": ts,
            "symbol": symbol,
            "close_price": daily.loc[ts],
            "circulating_supply": supply,
            "approx_market_cap": mcap,
        })
    print(f"  {symbol}: {len(daily)} daily records, supply={supply:,.0f}")

approx_df = pd.DataFrame(approx_records)
approx_df.to_parquet(APPROX_OUTPUT, index=False)
print(f"\nTotal records: {len(approx_df)}")
print(f"Date range: {approx_df['date'].min().date()} -> {approx_df['date'].max().date()}")
print(f"Saved: {APPROX_OUTPUT}")

Loaded circulating supply for 10 coins
  BTCUSDT: 3206 daily records, supply=20,023,521
  ETHUSDT: 3206 daily records, supply=120,687,385
  BNBUSDT: 3125 daily records, supply=134,785,930
  XRPUSDT: 2946 daily records, supply=61,796,225,236
  SOLUSDT: 2116 daily records, supply=576,326,280
  ADAUSDT: 2963 daily records, supply=36,974,724,813
  DOGEUSDT: 2519 daily records, supply=154,100,446,384
  AVAXUSDT: 2074 daily records, supply=431,771,961
  DOTUSDT: 2109 daily records, supply=1,682,240,546
  LINKUSDT: 2689 daily records, supply=727,099,970

Total records: 26953
Date range: 2017-08-17 -> 2026-05-27
Saved: /Users/wojciech.neuman/Documents/hybrid-multi-agent-trading-system/data/external/approx_market_caps.parquet


## Validation — CoinGecko vs Approximate Market Caps

Compare the overlap period to check if our static-supply approximation is reasonable.

In [6]:
cg = pd.read_parquet(CG_OUTPUT)
ap = pd.read_parquet(APPROX_OUTPUT)

cg["date_day"] = cg["date"].dt.normalize()
ap["date_day"] = ap["date"].dt.normalize()

for sym in ["BTCUSDT", "ETHUSDT"]:
    cg_sym = cg[cg["symbol"] == sym].set_index("date_day")["market_cap"].dropna()
    ap_sym = ap[ap["symbol"] == sym].set_index("date_day")["approx_market_cap"].dropna()

    overlap = cg_sym.index.intersection(ap_sym.index)
    if len(overlap) > 10:
        ratio = (ap_sym.loc[overlap] / cg_sym.loc[overlap]).dropna()
        print(f"{sym}: {len(overlap)} overlap days, approx/actual ratio = {ratio.mean():.3f} +/- {ratio.std():.3f}")
    else:
        print(f"{sym}: insufficient overlap ({len(overlap)} days)")

BTCUSDT: 364 overlap days, approx/actual ratio = 1.003 +/- 0.022
ETHUSDT: 364 overlap days, approx/actual ratio = 1.000 +/- 0.035


## Summary

In [7]:
print("=" * 60)
print("DATA INGESTION v1 — COMPLETE")
print("=" * 60)
print(f"\nOutput files:")
for f in sorted(EXT_DIR.glob("*.parquet")):
    size = f.stat().st_size / 1e6
    print(f"  {f.name:>40s}  {size:.2f} MB")

DATA INGESTION v1 — COMPLETE

Output files:
                approx_market_caps.parquet  0.40 MB
             coingecko_market_caps.parquet  0.14 MB
                  fear_greed_index.parquet  0.03 MB
